In [69]:
# https://learnopencv.com/fine-tuning-bert/

In [1]:
from pathlib import Path

import torch

from collections.abc import Callable

from datasets import Dataset, load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate
import glob
import numpy as np
import pandas as pd

from pathlib import Path


import sys

sys.path.insert(0, str(Path.cwd().parent))


from finetuning.commons import PipelineData, prepare_data, parse_pubtator, build_training_samples, samples_to_rels_like_df

/nix/store/vcapnvnswfafrsqa88vi4xip12wfghnd-python3-3.12.10-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# NOTE: these are the defaults might change according to avaibale VRAM
# -> BATCH SIZE and LR are halved if less than 8GB of VRAM is detected
BATCH_SIZE = 32
NUM_PROCS = 32
LR = 0.00005
EPOCHS = 5
MODEL = 'NeuML/pubmedbert-base-embeddings'
OUT_DIR = 'relations-bert'
CACHE_DIR = Path("cache")

PUBTATOR_FILE = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Dev.PubTator"



def load_or_cache(split: str, prefix: str, build: Callable[[], Dataset], use_cache: bool = True) -> Dataset:
    """Load a Hugging Face ``Dataset`` from disk cache or build and persist it.

    Cached data is stored under ``CACHE_DIR / f"{prefix}_{split}"`` (default: ``cache/``).
    On a cache hit, ``load_from_disk`` is used and ``build`` is not called.
    On a miss, ``build()`` runs once, the result is saved with ``save_to_disk``, then returned.

    Args:
        split: Split identifier used in the cache directory name (e.g. ``"train"``, ``"validation"``).
        prefix: Stage prefix distinguishing pipeline steps (e.g. ``"raw"``, ``"tokenized"``).
        build: Zero-argument callable that produces the dataset when the cache is missing.

    Returns:
        The dataset for the given split, either loaded from cache or freshly built.

    Examples:
        Download a Hub split and cache it as ``cache/raw_train/``::

            train = load_or_cache(
                "train",
                "raw",
                lambda: load_dataset("ccdv/arxiv-classification", split="train"),
            )

        Tokenize an in-memory split and cache as ``cache/tokenized_train/``::

            tokenized_train = load_or_cache(
                "train",
                "tokenized",
                lambda: train.map(preprocess_function, batched=True, batch_size=32),
            )

    Note:
        Delete the matching folder under ``cache/`` to force a rebuild after changing
        ``build``, the source data, or preprocessing.
    """
    cache_path = CACHE_DIR / f"{prefix}_{split}"
    if cache_path.exists() and use_cache:
        print(f"Loading {prefix} {split} from cache/")
        return load_from_disk(cache_path)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset = build()
    dataset.save_to_disk(cache_path)
    return dataset


In [3]:
MODE: str = "cpu"
if torch.backends.mps.is_available():
    MODE = "mps"
elif torch.cuda.is_available():
    MODE = "cuda"
else:
    print("No GPU or MPS available - uising CPU")

print(f"Using {MODE} for training")


hardware_specific_args = {}

if MODE == "mps":
    hardware_specific_args["fp16"] = False
    hardware_specific_args["dataloader_num_workers"] = 0
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE
    hardware_specific_args["learning_rate"] = LR
    

elif MODE == "cuda":
    # Check total GPU VRAM 
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA device VRAM: {total_vram_gb:.2f} GB")

    # Default: assume >8GB VRAM
    batch_div = 1
    lr_div = 1

    # 2070super only has 8gigs of VRAM :')
    if total_vram_gb <= 8.5:
        print("Detected ~8GB of VRAM or less, reducing batch size and learning rate.")
        batch_div = 2
        lr_div = 2

    hardware_specific_args["fp16"] = True
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["learning_rate"] = LR / lr_div

    print(f"Using {hardware_specific_args['per_device_train_batch_size']} for training")
    print(f"Using {hardware_specific_args['per_device_eval_batch_size']} for evaluation")
    print(f"Using {hardware_specific_args['learning_rate']} for learning rate")



Using cuda for training
CUDA device VRAM: 7.57 GB
Detected ~8GB of VRAM or less, reducing batch size and learning rate.
Using 16 for training
Using 16 for evaluation
Using 2.5e-05 for learning rate


In [42]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=EPOCHS,
    report_to='tensorboard',
    **hardware_specific_args,
)

In [43]:


meta_df, anns_df, rels_df = parse_pubtator(PUBTATOR_FILE)

samples = build_training_samples(meta_df, anns_df, rels_df)


samples = samples_to_rels_like_df(samples)
samples["prompt"] = samples.apply(
    lambda row: f"{row['entity_a_text']} -> {row['relation_type']} -> {row['entity_b_text']} \n{row['abstract']}",
    axis=1
)



from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(samples, test_size=0.1, random_state=42, stratify=None)
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
# print(train_dataset)
# print(valid_dataset)
# print(test_dataset)

{'pmid': '14510914', 'relation_type': 'Positive_Correlation', 'id_1': 'c|DEL|1314_1328|', 'id_2': 'D003409', 'entity_a_text': 'deletion of the coding sequence (nt 1314 through nt 1328)', 'entity_b_text': 'Congenital hypothyroidism', 'perturbation': 'gold', 'label': 1, 'abstract': "OBJECTIVE: Iodide transport defect (ITD) is a rare disorder characterised by an inability of the thyroid to maintain an iodide gradient across the basolateral membrane of thyroid follicular cells, that often results in congenital hypothyroidism. When present the defect is also found in the salivary glands and gastric mucosa and it has been shown to arise from abnormalities of the sodium/iodide symporter (NIS). PATIENT: We describe a woman with hypothyroidism identified at the 3rd month of life. The diagnosis of ITD was suspected because of nodular goitre, and little if any iodide uptake by the thyroid and salivary glands. Treatment with iodide partially corrected the hypothyroidism; however, long-term substit

In [44]:
train_dataset[0]

{'pmid': '26684240',
 'relation_type': 'Comparison',
 'id_1': '5921',
 'id_2': '7428',
 'entity_a_text': 'RASA1',
 'entity_b_text': 'VHL',
 'perturbation': 'false_positive',
 'label': 0,
 'abstract': 'Genomic profiles of gastroenteropancreatic neuroendocrine tumors (GEP-NETs) are still insufficiently understood, and the genetic alterations associated with drug responses have not been studied. Here, we performed whole exome sequencing of 12 GEP-NETs from patients enrolled in a nonrandomized, open-labeled, single-center phase II study for pazopanib, and integrated our results with previously published results on pancreas (n = 12) and small intestine NETs (n = 50). The mean numbers of somatic mutations in each case varied widely from 20 to 4682. Among 12 GEP-NETs, eight showed mutations of more than one cancer-related gene, including TP53, CNBD1, RB1, APC, BCOR, BRAF, CTNNB1, EGFR, EP300, ERBB3, KDM6A, KRAS, MGA, MLL3, PTEN, RASA1, SMARCB1, SPEN, TBC1D12, and VHL. TP53 was recurrently mut

In [45]:
train_dataset[0].keys()

dict_keys(['pmid', 'relation_type', 'id_1', 'id_2', 'entity_a_text', 'entity_b_text', 'perturbation', 'label', 'abstract', 'prompt'])

In [46]:
label2id: dict[str, int] = {"false": 0, "true": 1}
id2label: dict[int, str] = {0: "false", 1: "true"}

In [47]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [48]:
def preprocess_function(examples, key: str = "prompt"):
    return tokenizer(
        examples[key],
        truncation=True,
        padding=True,
        max_length=512,
    )


In [49]:
def _tokenize(dataset: Dataset) -> Dataset:
    return dataset.map(
        preprocess_function,
        batched=True,
        batch_size=BATCH_SIZE,
        num_proc=NUM_PROCS,
    )


tokenized_train = load_or_cache("train", "bio_red_tokenized", lambda: _tokenize(train_dataset), use_cache=False)
tokenized_valid = load_or_cache("valid", "bio_red_tokenized", lambda: _tokenize(valid_dataset), use_cache=False)
# tokenized_test = load_or_cache("test", "bio_red_tokenized", lambda: _tokenize(test_dataset))

Saving the dataset (1/1 shards): 100%|██████████| 503/503 [00:00<00:00, 35290.47 examples/s]


In [50]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [51]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)
print(f"Length of tokenized IDs: {len(tokenized_sample.input_ids)}")
print(f"Length of attention mask: {len(tokenized_sample.attention_mask)}")

{'input_ids': [2, 6520, 4904, 17, 34, 3723, 17, 34, 20348, 4935, 5554, 1927, 17634, 8228, 15895, 1936, 2479, 15980, 3861, 12, 3929, 1034, 17, 17419, 13, 2032, 3974, 9406, 1954, 7372, 16, 1930, 1920, 3299, 5996, 2458, 1956, 2838, 3387, 2162, 2084, 2252, 3942, 18, 3410, 16, 2038, 2593, 4292, 16390, 4754, 1927, 2369, 3929, 1034, 17, 17419, 2037, 2132, 7316, 1922, 43, 22060, 24922, 2301, 16, 4437, 17, 4993, 16, 2957, 17, 4102, 3193, 2890, 2161, 1958, 27878, 8228, 8635, 1014, 16, 1930, 6714, 2342, 2274, 1956, 3024, 4586, 2274, 1990, 10936, 12, 56, 33, 2369, 13, 1930, 2858, 9887, 17419, 12, 56, 33, 2761, 13, 18, 1920, 2667, 4630, 1927, 8548, 3527, 1922, 2362, 3087, 6760, 5673, 2037, 2036, 1942, 23175, 1028, 18, 2706, 2369, 3929, 1034, 17, 17419, 16, 5066, 2594, 3527, 1927, 2253, 2254, 2340, 2539, 17, 2712, 2359, 16, 2710, 13544, 16, 5940, 13602, 1009, 16, 24460, 16, 9187, 16, 6452, 1924, 16, 11346, 16, 30239, 5090, 16, 5775, 16, 2630, 17450, 16, 30359, 16, 21236, 11084, 16, 11083, 16, 2786, 

In [52]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)

{'input_ids': [2, 6520, 4904, 17, 34, 3723, 17, 34, 20348, 4935, 5554, 1927, 17634, 8228, 15895, 1936, 2479, 15980, 3861, 12, 3929, 1034, 17, 17419, 13, 2032, 3974, 9406, 1954, 7372, 16, 1930, 1920, 3299, 5996, 2458, 1956, 2838, 3387, 2162, 2084, 2252, 3942, 18, 3410, 16, 2038, 2593, 4292, 16390, 4754, 1927, 2369, 3929, 1034, 17, 17419, 2037, 2132, 7316, 1922, 43, 22060, 24922, 2301, 16, 4437, 17, 4993, 16, 2957, 17, 4102, 3193, 2890, 2161, 1958, 27878, 8228, 8635, 1014, 16, 1930, 6714, 2342, 2274, 1956, 3024, 4586, 2274, 1990, 10936, 12, 56, 33, 2369, 13, 1930, 2858, 9887, 17419, 12, 56, 33, 2761, 13, 18, 1920, 2667, 4630, 1927, 8548, 3527, 1922, 2362, 3087, 6760, 5673, 2037, 2036, 1942, 23175, 1028, 18, 2706, 2369, 3929, 1034, 17, 17419, 16, 5066, 2594, 3527, 1927, 2253, 2254, 2340, 2539, 17, 2712, 2359, 16, 2710, 13544, 16, 5940, 13602, 1009, 16, 24460, 16, 9187, 16, 6452, 1924, 16, 11346, 16, 30239, 5090, 16, 5775, 16, 2630, 17450, 16, 30359, 16, 21236, 11084, 16, 11083, 16, 2786, 

In [53]:
accuracy = evaluate.load('precision')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


In [54]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at NeuML/pubmedbert-base-embeddings and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
if MODE != "cpu":
    model = model.to(MODE)
print(f"Moving model to {MODE}")


Moving model to cuda


In [58]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [59]:
history = trainer.train()

Epoch,Training Loss,Validation Loss,Precision
1,No log,0.358126,0.786517
2,0.308100,0.279182,0.677852
3,0.308100,0.294382,0.697183
4,0.229600,0.346343,0.724409
5,0.229600,0.371852,0.705882


In [ ]:
	
trainer.evaluate(tokenized_valid)

{'eval_loss': 0.28094425797462463,
 'eval_precision': 0.7364341085271318,
 'eval_runtime': 3.6507,
 'eval_samples_per_second': 137.782,
 'eval_steps_per_second': 8.765,
 'epoch': 5.0}

In [ ]:
AutoModelForSequenceClassification.from_pretrained(f"arxiv_bert/checkpoint-3550")
 
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
classify = pipeline(task='text-classification', model=model, tokenizer=tokenizer)
 
all_files = glob.glob('arxiv_custom_inference_data/*')
for file_name in all_files:
    file = open(file_name)
    content = file.read()
    print(content)
    result = classify(content)
    print('PRED: ', result)
    print('GT: ', file_name.split('_')[-1].split('.txt')[0])
    print('\n')

Device set to use cuda:0


To every local complete intersection ring one may associate a so-called generic hypersurface. In this paper we introduce rank varieties for modules and complexes over the generic hypersurface. The definition uses extension of scalars, rather than restriction of scalars which are used to define the conventional support varieties over a local complete intersection. We show that every projective variety can be realized as the rank variety of a finitely generated module over the generic hypersurface. We also investigate several properties of these rank varieties. 
PRED:  [{'label': 'math.AC', 'score': 0.9941840767860413}]
GT:  math.AC


Current hierarchical attention methods, such as NSA and InfLLMv2, select the top-k relevant key-value (KV) blocks based on coarse attention scores and subsequently apply fine-grained softmax attention on the selected tokens. However, the top-k operation assumes the number of relevant tokens for any query is fixed and it precludes the gradient flow between t